# LSTM-GATv2 with Previous Window Attention (Step 1b)

Uses **raw features from the previous non-overlapping window** to compute
GATv2 attention, then applies that graph for message passing on the current
window's LSTM hidden states.

**Key innovation:**
- Attention (graph construction) from previous window raw features (200-dim)
- Message passing on current window LSTM hidden states (per timestep)
- No information leakage: graph is computed from strictly past data
- Analogous to rolling GCN (lookback correlation → graph), but learned end-to-end

**Toggle:** `ATTN_FEATURE_SOURCE` = `"straddle"` (200-dim) or `"equity_returns"` (20-dim)

## 1. Setup

In [ ]:
!pip install -q tensorflow>=2.16.0 keras-tuner empyrical-reloaded spektral

In [ ]:
import os
import sys

if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/repo'):
        !git clone https://github.com/adam-909/4yp.git /content/repo
    else:
        !cd /content/repo && git pull
    os.chdir('/content/repo/4YP-main')
else:
    os.chdir('/home/adam/new4YP/4YP-main')

sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from empyrical import (
    sharpe_ratio, sortino_ratio, max_drawdown,
    annual_return, annual_volatility, calmar_ratio,
)

import random
random.seed(40)
np.random.seed(40)

import tensorflow as tf
tf.random.set_seed(40)

print(f"TensorFlow version: {tf.__version__}")

## 2. Configuration

In [ ]:
# Training/Test Configuration
TRAIN_START = 2011
TEST_START = 2017
TEST_END = 2023

VOL_TARGET = 0.15
TRAIN_STRIDE = 20  # Non-overlapping independent windows

# Attention feature source
# "straddle"       : use all 10 straddle features from prev window (200-dim)
# "equity_returns"  : use equity log returns from prev window (20-dim, cleaner signal)
ATTN_FEATURE_SOURCE = "straddle"

# Model Configuration
TOTAL_TIME_STEPS = 20
TRAIN_VALID_RATIO = 0.8
NUM_EPOCHS = 300
EARLY_STOPPING_PATIENCE = 25

# GATv2 Hyperparameters
HIDDEN_LAYER_SIZE = 32
GAT_UNITS = 16
ATTN_HEADS = 4
LSTM_DROPOUT = 0.3
ATTN_DROPOUT = 0.1
LEARNING_RATE = 0.001
MAX_GRADIENT_NORM = 1.0
BATCH_SIZE = 128

print(f"Train: {TRAIN_START}-{TEST_START}")
print(f"Test:  {TEST_START}-{TEST_END}")
print(f"\nModel: LSTM-GATv2 Previous Window Attention (Step 1b)")
print(f"  Attention features: {ATTN_FEATURE_SOURCE}")
print(f"  Stride: {TRAIN_STRIDE}")
print(f"  LSTM hidden: {HIDDEN_LAYER_SIZE}, dropout: {LSTM_DROPOUT}")
print(f"  GAT units: {GAT_UNITS}, heads: {ATTN_HEADS}")
print(f"  Attention dropout: {ATTN_DROPOUT}")
print(f"  LR: {LEARNING_RATE}, clip norm: {MAX_GRADIENT_NORM}")
print(f"  Batch size: {BATCH_SIZE}")

## 3. Helper Functions

In [ ]:
def calc_daily_returns(df, returns_col='captured_returns'):
    num_tickers = df['identifier'].nunique()
    daily_ret = df.groupby('time')[returns_col].sum() / num_tickers
    daily_ret.index = pd.to_datetime(daily_ret.index)
    return daily_ret.sort_index()


def calc_vol_scaled_returns(daily_returns, target_vol=0.15):
    current_vol = daily_returns.std() * np.sqrt(252)
    if current_vol > 0:
        return daily_returns * (target_vol / current_vol)
    return daily_returns


def calc_metrics(daily_returns, name="Strategy"):
    return {
        'Strategy': name,
        'E[Ret.]': annual_return(daily_returns),
        'Vol.': annual_volatility(daily_returns),
        'Sharpe': sharpe_ratio(daily_returns),
        'Sortino': sortino_ratio(daily_returns),
        'Max DD': -max_drawdown(daily_returns),
        'Calmar': calmar_ratio(daily_returns),
        'Hit Rate': (daily_returns > 0).mean(),
        'Avg P/L': daily_returns[daily_returns > 0].mean() / abs(daily_returns[daily_returns < 0].mean()) if (daily_returns < 0).any() else np.nan,
    }


def calc_metrics_vol_normalized(daily_returns, name="Strategy", target_vol=0.15):
    scaled = calc_vol_scaled_returns(daily_returns, target_vol)
    return calc_metrics(scaled, name + " (Vol-Norm)"), scaled


def display_metrics(metrics_dict):
    df = pd.DataFrame([metrics_dict]).set_index('Strategy')
    for col in ['E[Ret.]', 'Vol.', 'Max DD', 'Hit Rate']:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: f"{x:.2%}")
    for col in ['Sharpe', 'Sortino', 'Calmar', 'Avg P/L']:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: f"{x:.3f}")
    display(df)
    return df


def calc_yearly_sharpes(daily_returns):
    yearly = {}
    for year in sorted(daily_returns.index.year.unique()):
        yr_ret = daily_returns[daily_returns.index.year == year]
        yearly[year] = sharpe_ratio(yr_ret)
    return yearly


def plot_results(daily_returns_dict, title="Strategy Comparison"):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    colors = plt.cm.tab10(np.linspace(0, 1, len(daily_returns_dict)))

    ax1 = axes[0, 0]
    for (name, returns), color in zip(daily_returns_dict.items(), colors):
        cum_ret = (1 + returns).cumprod() - 1
        ax1.plot(cum_ret.index, cum_ret.values, label=name, linewidth=1.5, color=color)
    ax1.set_title('Cumulative Returns')
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Cumulative Return')
    ax1.legend(loc='upper left', fontsize=8)
    ax1.grid(True, alpha=0.3)

    ax2 = axes[0, 1]
    for (name, returns), color in zip(daily_returns_dict.items(), colors):
        cum = (1 + returns).cumprod()
        running_max = cum.cummax()
        drawdown = (cum - running_max) / running_max
        ax2.fill_between(drawdown.index, drawdown.values, 0, alpha=0.3, label=name, color=color)
    ax2.set_title('Drawdown')
    ax2.set_xlabel('Date')
    ax2.set_ylabel('Drawdown')
    ax2.legend(loc='lower left', fontsize=8)
    ax2.grid(True, alpha=0.3)

    ax3 = axes[1, 0]
    for (name, returns), color in zip(daily_returns_dict.items(), colors):
        rolling_sharpe = returns.rolling(252).mean() / returns.rolling(252).std() * np.sqrt(252)
        ax3.plot(rolling_sharpe.index, rolling_sharpe.values, label=name, linewidth=1, color=color)
    ax3.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
    ax3.set_title('Rolling 252-Day Sharpe Ratio')
    ax3.set_xlabel('Date')
    ax3.set_ylabel('Sharpe Ratio')
    ax3.legend(loc='upper left', fontsize=8)
    ax3.grid(True, alpha=0.3)

    ax4 = axes[1, 1]
    yearly_data = {name: calc_yearly_sharpes(returns) for name, returns in daily_returns_dict.items()}
    yearly_df = pd.DataFrame(yearly_data)
    yearly_df.plot(kind='bar', ax=ax4, width=0.8)
    ax4.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
    ax4.set_title('Yearly Sharpe Ratios')
    ax4.set_xlabel('Year')
    ax4.set_ylabel('Sharpe Ratio')
    ax4.legend(loc='upper right', fontsize=8)
    ax4.tick_params(axis='x', rotation=45)
    ax4.grid(True, alpha=0.3, axis='y')

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 4. Data Loading

In [ ]:
features_path = "data/straddle_features/features.csv"
df = pd.read_csv(features_path)
df['date'] = pd.to_datetime(df['date'])

print(f"Loaded {len(df)} rows")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Tickers: {df['ticker'].nunique()}")

In [ ]:
from gml.graph_model_inputs import GraphModelFeatures

features = GraphModelFeatures(
    df=df,
    total_time_steps=TOTAL_TIME_STEPS,
    start_boundary=TRAIN_START,
    test_boundary=TEST_START,
    test_end=TEST_END,
    train_valid_ratio=TRAIN_VALID_RATIO,
    split_tickers_individually=True,
    time_features=False,
    train_valid_sliding=True,
)

print("Feature generator created.")

In [ ]:
# Apply stride for non-overlapping windows
train_all = {k: v[::TRAIN_STRIDE] for k, v in features.train.items()}
valid_all = {k: v[::TRAIN_STRIDE] for k, v in features.valid.items()}
test_all = features.test_sliding  # stride=1 for test

print(f"Before pairing (stride={TRAIN_STRIDE}):")
print(f"  Train: {train_all['inputs'].shape[0]} windows")
print(f"  Valid: {valid_all['inputs'].shape[0]} windows")
print(f"  Test:  {test_all['inputs'].shape[0]} windows")

## 5. Pair Consecutive Windows (Previous → Current)

In [ ]:
def pair_consecutive_windows(data, input_size, attn_feature_source="straddle",
                             equity_returns_pivot=None, date_key='date'):
    """
    Pair consecutive non-overlapping windows for previous-window attention.

    Returns:
        prev_features: (N-1, num_tickers, prev_feature_dim) - flattened prev window
        curr_features: (N-1, num_tickers, time_steps, input_size) - current window
        curr_outputs:  (N-1, num_tickers, time_steps, 1) - labels for current window
        curr_weights:  (N-1, num_tickers, time_steps, 1) - active entries
    """
    inputs = data['inputs']    # (N, tickers, time, features)
    outputs = data['outputs']  # (N, tickers, time, 1)
    active = data['active_entries']
    N = inputs.shape[0]
    num_tickers = inputs.shape[1]
    time_steps = inputs.shape[2]

    if attn_feature_source == "straddle":
        # Flatten previous window's straddle features: (N, tickers, time*features)
        prev_flat = inputs[:-1].reshape(N - 1, num_tickers, -1)
        prev_feature_dim = time_steps * input_size
    elif attn_feature_source == "equity_returns":
        if equity_returns_pivot is None:
            raise ValueError(
                "equity_returns_pivot must be provided when "
                "attn_feature_source='equity_returns'"
            )
        # For each previous window, extract equity returns for those dates
        dates = data[date_key]  # (N, tickers, time, 1)
        tickers = sorted(equity_returns_pivot.columns.tolist())

        prev_features_list = []
        for w in range(N - 1):
            # Get dates for this previous window (use ticker 0's dates)
            window_dates = pd.to_datetime(dates[w, 0, :, 0])
            # Extract equity returns for these dates
            equity_window = equity_returns_pivot.reindex(window_dates).fillna(0.0)
            # Shape: (time_steps, num_tickers) → transpose to (num_tickers, time_steps)
            equity_arr = equity_window[tickers[:num_tickers]].values.T
            prev_features_list.append(equity_arr)

        prev_flat = np.array(prev_features_list)  # (N-1, tickers, time_steps)
        prev_feature_dim = time_steps
    else:
        raise ValueError(f"Unknown attn_feature_source: {attn_feature_source}")

    curr_features = inputs[1:]    # (N-1, tickers, time, features)
    curr_outputs = outputs[1:]    # (N-1, tickers, time, 1)
    curr_weights = active[1:]     # (N-1, tickers, time, 1)

    print(f"Paired {N} windows → {N-1} (prev, curr) pairs")
    print(f"  prev_features: {prev_flat.shape} (dim={prev_feature_dim})")
    print(f"  curr_features: {curr_features.shape}")

    return prev_flat, curr_features, curr_outputs, curr_weights, prev_feature_dim

In [ ]:
# Load equity returns if needed
equity_returns_pivot = None
if ATTN_FEATURE_SOURCE == "equity_returns":
    cache_path = os.path.join("data", "graph_structure", "equity_returns", "log_returns.csv")
    if os.path.exists(cache_path):
        equity_returns_pivot = pd.read_csv(cache_path, index_col=0, parse_dates=True)
        equity_returns_pivot = equity_returns_pivot.sort_index()
        print(f"Loaded equity returns: {equity_returns_pivot.shape}")
    else:
        raise FileNotFoundError(
            f"Equity returns cache not found at {cache_path}. "
            "Run 'python examples/create_equity_returns_cache.py' to create it."
        )

input_size = train_all['inputs'].shape[3]

# Pair training windows
print("\nTraining:")
prev_train, curr_train, y_train, w_train, prev_feature_dim = pair_consecutive_windows(
    train_all, input_size, ATTN_FEATURE_SOURCE, equity_returns_pivot
)

# Pair validation windows
print("\nValidation:")
prev_valid, curr_valid, y_valid, w_valid, _ = pair_consecutive_windows(
    valid_all, input_size, ATTN_FEATURE_SOURCE, equity_returns_pivot
)

# Pair test windows
print("\nTest:")
prev_test, curr_test, y_test, w_test, _ = pair_consecutive_windows(
    test_all, input_size, ATTN_FEATURE_SOURCE, equity_returns_pivot
)

print(f"\nAttention feature source: {ATTN_FEATURE_SOURCE}")
print(f"Previous window feature dim: {prev_feature_dim}")

## 6. Model Definition

In [ ]:
from gml.graph_attention_v2 import build_lstm_gat_e2e_v2_prev_window

num_tickers = curr_train.shape[1]
time_steps = curr_train.shape[2]

print(f"Building LSTM-GATv2 Previous Window model (Step 1b):")
print(f"  num_tickers: {num_tickers}")
print(f"  time_steps: {time_steps}")
print(f"  input_size: {input_size}")
print(f"  prev_feature_dim: {prev_feature_dim}")

model = build_lstm_gat_e2e_v2_prev_window(
    num_tickers=num_tickers,
    time_steps=time_steps,
    input_size=input_size,
    hidden_layer_size=HIDDEN_LAYER_SIZE,
    gat_units=GAT_UNITS,
    attn_heads=ATTN_HEADS,
    lstm_dropout=LSTM_DROPOUT,
    attn_dropout=ATTN_DROPOUT,
    learning_rate=LEARNING_RATE,
    max_gradient_norm=MAX_GRADIENT_NORM,
    prev_feature_dim=prev_feature_dim,
)

print(f"\nTotal parameters: {model.count_params():,}")

## 7. Training

In [ ]:
print(f"Training samples: {curr_train.shape[0]}")
print(f"Validation samples: {curr_valid.shape[0]}")

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=EARLY_STOPPING_PATIENCE,
    restore_best_weights=True,
    verbose=1
)

print("=" * 60)
print("Training LSTM-GATv2 Previous Window (Step 1b)")
print(f"  Attn source: {ATTN_FEATURE_SOURCE} (dim={prev_feature_dim})")
print(f"  LSTM hidden: {HIDDEN_LAYER_SIZE}, GAT: {GAT_UNITS}x{ATTN_HEADS}")
print(f"  Dropout: LSTM={LSTM_DROPOUT}, Attn={ATTN_DROPOUT}")
print(f"  LR: {LEARNING_RATE}, clip: {MAX_GRADIENT_NORM}, batch: {BATCH_SIZE}")
print("=" * 60)

history = model.fit(
    [prev_train, curr_train], y_train,
    sample_weight=w_train,
    validation_data=([prev_valid, curr_valid], y_valid, w_valid),
    epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=1,
)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (Negative Sharpe)')
plt.title('LSTM-GATv2 Previous Window (Step 1b) Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Epochs trained: {len(history.history['loss'])}")
print(f"Best val loss: {min(history.history['val_loss']):.4f}")

## 8. Evaluation

In [ ]:
predictions = model.predict([prev_test, curr_test])
print(f"Predictions shape: {predictions.shape}")

In [ ]:
# Extract last timestep
positions = predictions[:, :, -1, 0].reshape(-1)
returns = y_test[:, :, -1, 0].reshape(-1)
captured_returns = positions * returns

# Dates from test_all (offset by 1 since we paired consecutive)
test_dates = test_all['date'][1:]  # skip first window (no previous)
test_ids = test_all['identifier'][1:]

dates = test_dates[:, :, -1, 0].reshape(-1)
identifiers = test_ids[:, :, -1, 0].reshape(-1)

results_df = pd.DataFrame({
    'time': dates, 'identifier': identifiers,
    'position': positions, 'returns': returns,
    'captured_returns': captured_returns,
})

results_df['time'] = pd.to_datetime(results_df['time'])
results_df = results_df[results_df['identifier'] != '0']

print(f"Results: {len(results_df)} rows")
results_df.head()

In [ ]:
daily_returns = calc_daily_returns(results_df)

print("\n" + "=" * 60)
print(f"LSTM-GATv2 Prev Window ({ATTN_FEATURE_SOURCE}) Results (Raw)")
print("=" * 60)

metrics_raw = calc_metrics(daily_returns, f"GATv2 PrevWin ({ATTN_FEATURE_SOURCE})")
display_metrics(metrics_raw)

print(f"\nVolatility-Normalized (Target: {VOL_TARGET:.0%})")
metrics_norm, scaled_returns = calc_metrics_vol_normalized(
    daily_returns, f"GATv2 PrevWin ({ATTN_FEATURE_SOURCE})", VOL_TARGET
)
display_metrics(metrics_norm)

print("\nYearly Sharpe Ratios:")
yearly_sharpes = calc_yearly_sharpes(daily_returns)
for year, sharpe_val in yearly_sharpes.items():
    print(f"  {year}: {sharpe_val:.4f}")

In [ ]:
all_daily_returns = {f'GATv2 PrevWin ({ATTN_FEATURE_SOURCE})': daily_returns}
plot_results(all_daily_returns, f"LSTM-GATv2 Previous Window ({TEST_START}-{TEST_END})")

## 9. Attention Analysis & Entropy

In [ ]:
# Extract attention weights from the PrevWindowAttentionLayer
# We need to manually compute attention from the prev_features
gat_layer = model.get_layer("prev_window_gat")
W_src = gat_layer.W_src.numpy()
W_dst = gat_layer.W_dst.numpy()
a = gat_layer.a.numpy()
num_heads = W_src.shape[0]

def compute_attention_from_prev(prev_features, W_src, W_dst, a):
    """Compute attention weights from previous window features."""
    num_heads = W_src.shape[0]
    N = prev_features.shape[0]  # num samples

    all_attn = []
    for idx in range(N):
        sample = prev_features[idx]  # (nodes, prev_dim)
        head_attns = []
        for h in range(num_heads):
            h_src = sample @ W_src[h]  # (nodes, units)
            h_dst = sample @ W_dst[h]
            pairwise = h_src[:, np.newaxis, :] + h_dst[np.newaxis, :, :]
            pairwise = np.where(pairwise > 0, pairwise, 0.2 * pairwise)
            scores = (pairwise @ a[h]).squeeze(-1)  # (nodes, nodes)
            exp_s = np.exp(scores - scores.max(axis=-1, keepdims=True))
            attn = exp_s / (exp_s.sum(axis=-1, keepdims=True) + 1e-9)
            head_attns.append(attn)
        all_attn.append(np.stack(head_attns, axis=0))

    return np.array(all_attn)  # (N, heads, nodes, nodes)

# Compute for test set
all_graphs = compute_attention_from_prev(prev_test, W_src, W_dst, a)
print(f"Extracted attention: {all_graphs.shape}")

# Average across heads
all_graphs_avg = all_graphs.mean(axis=1)  # (N, nodes, nodes)

# Entropy
entropy = -np.sum(all_graphs_avg * np.log(all_graphs_avg + 1e-9), axis=-1)
mean_entropy = entropy.mean()
max_entropy = np.log(all_graphs_avg.shape[-1])
print(f"\nAttention entropy: {mean_entropy:.3f} (uniform = {max_entropy:.3f})")
print(f"Entropy ratio: {mean_entropy / max_entropy:.3f}")

In [ ]:
# Heatmap + entropy distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

avg_attn = all_graphs_avg.mean(axis=0)
sns.heatmap(avg_attn, ax=axes[0], cmap='viridis', vmin=0)
axes[0].set_title('Average Learned Attention (all test windows)')
axes[0].set_xlabel('Target Stock')
axes[0].set_ylabel('Source Stock')

axes[1].hist(entropy.flatten(), bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(x=max_entropy, color='red', linestyle='--', label=f'Uniform = {max_entropy:.2f}')
axes[1].axvline(x=mean_entropy, color='blue', linestyle='--', label=f'Mean = {mean_entropy:.2f}')
axes[1].set_xlabel('Entropy (nats)')
axes[1].set_ylabel('Count')
axes[1].set_title('Attention Entropy Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Interactive Graph Visualization

In [ ]:
import networkx as nx
import ipywidgets as widgets
from IPython.display import display, clear_output
from settings.default import ALL_TICKERS, BBG_SECTORS

SECTOR_COLORS = {
    "Information Technology": "#1f77b4",
    "Healthcare": "#2ca02c",
    "Financials": "#ff7f0e",
    "Consumer Discretionary": "#d62728",
    "Consumer Staples": "#9467bd",
    "Industrials": "#8c564b",
    "Communication Services": "#e377c2",
    "Energy": "#7f7f7f",
    "Utilities": "#bcbd22",
    "Real Estate": "#17becf",
}

tickers = sorted(ALL_TICKERS)
G_ref = nx.Graph()
for t in tickers:
    G_ref.add_node(t, sector=BBG_SECTORS.get(t, "Unknown"))

fixed_pos = nx.spring_layout(G_ref, k=2.5, iterations=100, seed=42)
node_colors = [SECTOR_COLORS.get(BBG_SECTORS.get(t, "Unknown"), "#cccccc") for t in tickers]

# Rolling Sharpe for bottom panel
rolling_sharpe = daily_returns.rolling(252).mean() / daily_returns.rolling(252).std() * np.sqrt(252)

# Test dates (offset by 1 for pairing)
test_dates_arr = pd.to_datetime(test_all['date'][1:, 0, -1, 0])

print(f"Ready: {len(all_graphs_avg)} graphs, {len(test_dates_arr)} dates")

In [ ]:
EDGE_THRESHOLD = 0.02  # Attention weight threshold for showing edges
# Note: 1/88 = 0.0114, so 0.02 means "above uniform"

output_widget = widgets.Output()

def update_graph(window_idx):
    with output_widget:
        clear_output(wait=True)

        fig, (ax_graph, ax_sharpe) = plt.subplots(
            2, 1, figsize=(14, 16),
            gridspec_kw={'height_ratios': [3, 1]}
        )

        # --- Top: Learned attention graph ---
        adj = all_graphs_avg[window_idx]
        date_str = str(test_dates_arr[window_idx].date()) if window_idx < len(test_dates_arr) else "N/A"

        G = nx.Graph()
        for t in tickers:
            G.add_node(t)

        n = len(tickers)
        for i in range(n):
            for j in range(i + 1, n):
                w = (adj[i, j] + adj[j, i]) / 2  # symmetrize
                if w > EDGE_THRESHOLD:
                    G.add_edge(tickers[i], tickers[j], weight=w)

        edges = G.edges(data=True)
        if len(edges) > 0:
            weights = [d['weight'] for _, _, d in edges]
            max_w = max(weights) if weights else 1.0
            for (u, v, d) in edges:
                w = d['weight']
                width = 2.0 * w / max_w
                alpha = 0.3 + 0.7 * w / max_w
                x = [fixed_pos[u][0], fixed_pos[v][0]]
                y = [fixed_pos[u][1], fixed_pos[v][1]]
                ax_graph.plot(x, y, color='gray', linewidth=width, alpha=alpha, zorder=1)

        nx.draw_networkx_nodes(G, fixed_pos, node_color=node_colors,
                               node_size=600, alpha=0.9, ax=ax_graph)
        nx.draw_networkx_labels(G, fixed_pos, font_size=6, font_weight='bold', ax=ax_graph)

        num_edges = G.number_of_edges()
        ax_graph.set_title(
            f"Learned Attention Graph at {date_str}  |  {num_edges} edges (threshold={EDGE_THRESHOLD})",
            fontsize=14, fontweight='bold'
        )
        ax_graph.axis('off')

        # --- Bottom: Rolling Sharpe with date marker ---
        ax_sharpe.plot(rolling_sharpe.index, rolling_sharpe.values, linewidth=1, color='blue')
        ax_sharpe.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
        if window_idx < len(test_dates_arr):
            ax_sharpe.axvline(x=test_dates_arr[window_idx], color='red', linewidth=2, alpha=0.8)
        ax_sharpe.set_title('Rolling 252-Day Sharpe Ratio', fontsize=12)
        ax_sharpe.set_xlabel('Date')
        ax_sharpe.set_ylabel('Sharpe')
        ax_sharpe.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

slider = widgets.IntSlider(
    min=0, max=len(all_graphs_avg) - 1, step=1, value=0,
    description='Window:',
    continuous_update=False,
    layout=widgets.Layout(width='80%')
)

widgets.interactive(update_graph, window_idx=slider)
display(slider, output_widget)
update_graph(0)

## 11. Position Analysis

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(results_df['position'], bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Position')
plt.ylabel('Frequency')
plt.title('Position Distribution')
plt.axvline(x=0, color='red', linestyle='--', linewidth=1)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Save Results

In [ ]:
results_dir = f"results/lstm_gat_e2e_v2_prev_window_{ATTN_FEATURE_SOURCE}/{TEST_START}-{TEST_END}"
os.makedirs(results_dir, exist_ok=True)

results_df.to_csv(os.path.join(results_dir, "captured_returns_sw.csv"), index=False)
pd.DataFrame([metrics_raw]).to_csv(os.path.join(results_dir, "metrics_raw.csv"), index=False)
pd.DataFrame([metrics_norm]).to_csv(os.path.join(results_dir, "metrics_vol_normalized.csv"), index=False)
pd.DataFrame(yearly_sharpes.items(), columns=['Year', 'Sharpe']).to_csv(
    os.path.join(results_dir, "yearly_sharpes.csv"), index=False
)

print(f"Results saved to: {results_dir}")

## 13. Summary

In [ ]:
print("=" * 60)
print("EXPERIMENT SUMMARY")
print("=" * 60)
print(f"\nModel: LSTM-GATv2 Previous Window Attention (Step 1b)")
print(f"  Attention from previous window raw features ({ATTN_FEATURE_SOURCE})")
print(f"  No information leakage (graph from strictly past data)")
print(f"\nHyperparameters:")
print(f"  LSTM hidden: {HIDDEN_LAYER_SIZE}, dropout: {LSTM_DROPOUT}")
print(f"  GAT units: {GAT_UNITS}, heads: {ATTN_HEADS}")
print(f"  Attn dropout: {ATTN_DROPOUT}")
print(f"  Attn feature dim: {prev_feature_dim}")
print(f"  LR: {LEARNING_RATE}, clip norm: {MAX_GRADIENT_NORM}")
print(f"  Batch size: {BATCH_SIZE}, stride: {TRAIN_STRIDE}")
print(f"\nTraining: {TRAIN_START}-{TEST_START}")
print(f"Test:     {TEST_START}-{TEST_END}")
print(f"\nPerformance (Raw):")
print(f"  Sharpe: {metrics_raw['Sharpe']:.3f}")
print(f"  Return: {metrics_raw['E[Ret.]']:.2%}")
print(f"  Vol:    {metrics_raw['Vol.']:.2%}")
print(f"  Sortino: {metrics_raw['Sortino']:.3f}")
print(f"  Max DD: {metrics_raw['Max DD']:.2%}")
print(f"\nAttention entropy: {mean_entropy:.3f} / {max_entropy:.3f} (ratio: {mean_entropy/max_entropy:.3f})")